In [6]:
from pathlib import Path

# project-relative path first:
INPUT_CSV = Path("data") / "lsu_hypoxia_2024.csv"

# Fallback for this environment (and any place you mounted the file elsewhere):
if not INPUT_CSV.exists():
    INPUT_CSV = Path("/mnt/data/lsu_hypoxia_2024.csv")

print("Using INPUT_CSV:", INPUT_CSV.resolve())

Using INPUT_CSV: /mnt/data/lsu_hypoxia_2024.csv


In [7]:
import pandas as pd
field = pd.read_csv(INPUT_CSV, encoding="ISO-8859-1").copy()
print("Rows:", len(field))

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/lsu_hypoxia_2024.csv'

In [8]:
from pathlib import Path
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [1]:
# ----------------------------
# PACE Level-3 matchups
# ----------------------------
# Reads ./data/lsu_hypoxia_2024.csv
# Pulls a matching PACE L3 mapped product per field date
# Extracts a variable at station location (nearest + 3×3 median)
# Writes ./output/pace_l3_matchups.csv

from __future__ import annotations

import os
from datetime import datetime, timedelta, timezone, date
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import earthaccess
import xarray as xr

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
CACHE_DIR = OUTPUT_DIR / "pace_l3_cache"
INPUT_CSV = DATA_DIR / "lsu_hypoxia_2024.csv"
OUTPUT_CSV = OUTPUT_DIR / "pace_l3_matchups.csv"

FORCE_L3_SHORTNAME = None  # e.g., "PACE_OCI_L3M_BGC"
L3_SHORTNAME_FALLBACKS = ["PACE_OCI_L3M_BGC", "PACE_OCI_L3M_AOP", "PACE_OCI_L3M_RRS", "PACE_OCI_L3M_CHL"]
VAR_CANDIDATES = ["chlor_a", "chl", "chl_ocx", "nflh", "avw"]
SEARCH_DAY_BUFFER_HOURS = 0

In [2]:
#helper functions

def ensure_dirs() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)


def require_env_creds() -> None:
    if not os.environ.get("EARTHDATA_USERNAME") or not os.environ.get("EARTHDATA_PASSWORD"):
        raise RuntimeError("Missing EARTHDATA_USERNAME/EARTHDATA_PASSWORD in environment.")


def parse_field_datetime_utc(date_str: str, time_str: str) -> datetime:
    dt_naive = datetime.strptime(f"{date_str} {time_str}", "%m/%d/%Y %H:%M")
    return dt_naive.replace(tzinfo=timezone.utc)


def day_window_utc(d: date) -> Tuple[str, str]:
    start = datetime(d.year, d.month, d.day, 0, 0, 0, tzinfo=timezone.utc) - timedelta(hours=SEARCH_DAY_BUFFER_HOURS)
    end = datetime(d.year, d.month, d.day, 23, 59, 59, tzinfo=timezone.utc) + timedelta(hours=SEARCH_DAY_BUFFER_HOURS)
    return start.isoformat(), end.isoformat()


def station_bbox(lat: float, lon: float, half_width_deg: float = 2.0) -> Tuple[float, float, float, float]:
    return (lon - half_width_deg, lat - half_width_deg, lon + half_width_deg, lat + half_width_deg)


def find_var_name(ds: xr.Dataset, candidates: List[str]) -> Optional[str]:
    for name in candidates:
        if name in ds.variables:
            return name
    lowered = {v.lower(): v for v in ds.variables}
    for cand in candidates:
        if cand.lower() in lowered:
            return lowered[cand.lower()]
    return None


def discover_pace_l3_shortname() -> Optional[str]:
    try:
        collections = list(earthaccess.search_datasets(keyword="PACE OCI Level-3"))
    except Exception:
        collections = []

    for c in collections:
        umm = getattr(c, "umm", None)
        if not isinstance(umm, dict):
            continue
        short = umm.get("ShortName")
        if not short or "PACE_OCI" not in short or "L3" not in short:
            continue
        pl = umm.get("ProcessingLevel", {})
        pl_id = pl.get("Id") if isinstance(pl, dict) else None
        if pl_id in (None, "3"):
            return short
    return None


def open_l3_dataset(file_obj: Any) -> xr.Dataset:
    for engine in ("h5netcdf", "netcdf4"):
        try:
            return xr.open_dataset(file_obj, engine=engine)
        except Exception:
            continue
    return xr.open_dataset(file_obj)


def extract_l3_nearest_and_3x3(ds: xr.Dataset, lat: float, lon: float, var_name: str) -> Dict[str, Any]:
    lat_name = "lat" if "lat" in ds.coords else ("latitude" if "latitude" in ds.coords else None)
    lon_name = "lon" if "lon" in ds.coords else ("longitude" if "longitude" in ds.coords else None)
    if lat_name is None or lon_name is None:
        raise ValueError("Could not find lat/lon coordinates in this L3 dataset.")

    lats = ds[lat_name].values
    lons = ds[lon_name].values

    lon_query = lon
    if np.nanmax(lons) > 180 and lon_query < 0:
        lon_query = lon_query % 360.0

    i_lat = int(np.argmin(np.abs(lats - lat)))
    i_lon = int(np.argmin(np.abs(lons - lon_query)))

    da = ds[var_name]
    arr = da.values
    dims = list(da.dims)

    if lat_name in dims and lon_name in dims:
        lat_axis = dims.index(lat_name)
        lon_axis = dims.index(lon_name)
    else:
        lat_axis, lon_axis = 0, 1

    def get_value(ii_lat: int, ii_lon: int) -> float:
        idx = [slice(None)] * arr.ndim
        idx[lat_axis] = ii_lat
        idx[lon_axis] = ii_lon
        v = arr[tuple(idx)]
        if np.ndim(v) == 0:
            return float(v)
        return float(np.ravel(v)[0])

    nearest_val = get_value(i_lat, i_lon)
    nearest_valid = bool(np.isfinite(nearest_val))

    lat0, lat1 = max(i_lat - 1, 0), min(i_lat + 2, len(lats))
    lon0, lon1 = max(i_lon - 1, 0), min(i_lon + 2, len(lons))

    idx_win = [slice(None)] * arr.ndim
    idx_win[lat_axis] = slice(lat0, lat1)
    idx_win[lon_axis] = slice(lon0, lon1)
    win = np.array(arr[tuple(idx_win)])

    if win.ndim > 2:
        win = win.reshape((win.shape[lat_axis], win.shape[lon_axis], -1))[..., 0]

    n_valid = int(np.sum(np.isfinite(win)))
    med_3x3 = float(np.nanmedian(win)) if n_valid > 0 else np.nan

    return {
        "pace_l3_var": var_name,
        "pace_l3_nearest": float(nearest_val) if nearest_valid else np.nan,
        "pace_l3_3x3_median": med_3x3,
        "pace_l3_nearest_valid": nearest_valid,
        "pace_l3_n_valid_3x3": n_valid,
        "pace_l3_grid_lat": float(lats[i_lat]),
        "pace_l3_grid_lon": float(lons[i_lon]),
        "pace_l3_i_lat": i_lat,
        "pace_l3_i_lon": i_lon,
    }



In [5]:
# ----------------------------
# Run
# ----------------------------
ensure_dirs()
require_env_creds()
earthaccess.login(strategy="environment")

field = pd.read_csv(INPUT_CSV, encoding="ISO-8859-1").copy()

col_date = "Date"
col_time = "Time CTD Cast began (UTC)"
col_station = "Station"
col_lat = "Lat"
col_lon = "Long"
col_do = "Corrected DO (mg/L)"
col_sal = "Surface Sal (psu)"
col_chl = "Surface Chl ( µg/L)"

field["field_time_utc"] = [parse_field_datetime_utc(d, t) for d, t in zip(field[col_date].astype(str), field[col_time].astype(str))]
field["field_date_utc"] = [dt.date() for dt in field["field_time_utc"]]

short_name = FORCE_L3_SHORTNAME or discover_pace_l3_shortname() or L3_SHORTNAME_FALLBACKS[0]

opened_by_date: Dict[date, List[Any]] = {}
rows_out: List[Dict[str, Any]] = []

for _, r in field.iterrows():
    station = str(r[col_station])
    lat = float(r[col_lat])
    lon = float(r[col_lon])
    ft = r["field_time_utc"]
    fd = r["field_date_utc"]

    base = {
        "Station": station,
        "field_time_utc": ft.isoformat(),
        "field_date_utc": fd.isoformat(),
        "field_lat": lat,
        "field_lon": lon,
        "field_surface_chl_ugL": float(r[col_chl]),
        "field_surface_sal_psu": float(r[col_sal]),
        "field_corrected_do_mgL": float(r[col_do]),
    }

    if fd not in opened_by_date:
        t0, t1 = day_window_utc(fd)
        bbox = station_bbox(lat, lon, half_width_deg=2.0)

        results: List[Any] = []
        tried = [short_name] + [sn for sn in L3_SHORTNAME_FALLBACKS if sn != short_name]

        for sn in tried:
            try:
                results = list(earthaccess.search_data(short_name=sn, temporal=(t0, t1), bounding_box=bbox))
                if results:
                    short_name = sn
                    break
            except Exception:
                continue

        opened_by_date[fd] = list(earthaccess.open(results)) if results else []

    files = opened_by_date.get(fd, [])
    if not files:
        rows_out.append({**base, "pace_l3_found": False, "pace_l3_short_name": short_name, "pace_l3_granule_count_day": 0, "error": None})
        continue

    best = None
    last_error = None

    for fobj in files:
        try:
            ds = open_l3_dataset(fobj)
            var = find_var_name(ds, VAR_CANDIDATES)
            if var is None:
                last_error = "No candidate variable found (update VAR_CANDIDATES)."
                continue
            ex = extract_l3_nearest_and_3x3(ds, lat, lon, var)
            best = {**base, "pace_l3_found": True, "pace_l3_short_name": short_name, "pace_l3_granule_count_day": len(files), **ex, "error": None}
            break
        except Exception as e:
            last_error = str(e)

    rows_out.append(best if best is not None else {**base, "pace_l3_found": False, "pace_l3_short_name": short_name, "pace_l3_granule_count_day": len(files), "error": last_error})

out = pd.DataFrame(rows_out)
out.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote: {OUTPUT_CSV.resolve()}")
out.head(10)

FileNotFoundError: [Errno 2] No such file or directory: 'data/lsu_hypoxia_2024.csv'

In [4]:
import os
import getpass
import earthaccess

os.environ["EARTHDATA_USERNAME"] = input("Earthdata username: ").strip()
os.environ["EARTHDATA_PASSWORD"] = getpass.getpass("Earthdata password: ")

earthaccess.login(strategy="environment")


Earthdata username:  ralfriedel
Earthdata password:  ········
